# Calibrate an LLM judge before you trust its error score

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/calibrate-an-ai-judge-before-you-trust-its-error-score/judge_calibration_v1.ipynb)

Companion to the article at https://profrod.ai/articles/calibrate-an-ai-judge-before-you-trust-its-error-score.

The run on Claude, 2026-09-30: 500 GSM8K test problems. Claude Haiku 4.5 answered each with a bare number and no working, so some answers are right and some wrong, and the GSM8K reference says which. Two judges (Haiku 4.5 and Sonnet 5.5) then graded every answer **without the reference**, once asked for a one-word verdict and once asked to work the problem first.

Part 1 recomputes every number in the article from the recorded results (no key needed). Part 2 reruns a small version of the experiment on your own Claude key.

In [ ]:
# Fetch the statistics module and the recorded run (skipped when run from a clone of the repository).
import json
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/calibrate-an-ai-judge-before-you-trust-its-error-score/"
FILES = [
    "judge_stats_v1.py",
    "judge_calibration_v1.py",
    "data/gsm8k-test-sample-v1.jsonl",
    "results/candidates-v1.jsonl",
    "results/judgments-v1.jsonl",
    "results/summary-v1.json",
    "receipts/judge-calibration-v1.json",
]
for name in FILES:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
print("ready:", len(FILES), "files")

## Part 1: recompute the article's numbers

Ground truth is exact: an answer is wrong unless it equals the GSM8K reference. A *positive* is a wrong answer, the thing a judge exists to catch.

In [ ]:
import judge_stats_v1 as S


def rows(path):
    return [json.loads(line) for line in open(path) if line.strip()]


cands = {c["id"]: c for c in rows("results/candidates-v1.jsonl")}
judgments = rows("results/judgments-v1.jsonl")
ids = sorted(cands)
truth = [cands[i]["truthWrong"] for i in ids]
print(f"{len(ids)} answers, {sum(truth)} wrong ({sum(truth) / len(ids):.1%})")

for judge in ["claude-haiku-4-5-20251001", "claude-sonnet-5-5"]:
    for condition in ["direct", "reasoned"]:
        got = {j["id"]: j["verdict"] for j in judgments if j["judge"] == judge and j["condition"] == condition}
        c = S.confusion(truth, [got[i] == "INCORRECT" for i in ids])
        lo_w, hi_w = S.wilson(c.tp, c.tp + c.fn)
        lo_r, hi_r = S.wilson(c.tn, c.tn + c.fp)
        reported = (c.tp + c.fp) / c.n
        print(
            f"{judge[:16]:16s} {condition:8s} catches wrong {c.tp}/{c.tp + c.fn} [{lo_w:.3f}, {hi_w:.3f}]  "
            f"passes right {c.tn}/{c.tn + c.fp} [{lo_r:.3f}, {hi_r:.3f}]  kappa {c.kappa():.3f}  reports {reported:.1%} wrong"
        )

## The correction: recover the truth from a judge's report

A judge that catches a share $s_e$ of wrong answers and passes a share $s_p$ of right ones reports an error rate

$$r = \pi s_e + (1-\pi)(1-s_p),$$

so, once $s_e$ and $s_p$ are measured on labeled data, the true rate is

$$\pi = \frac{r + s_p - 1}{s_e + s_p - 1}.$$

On the same items this is an identity, so the test that matters is out of sample: measure $s_e$ and $s_p$ on one half, correct the other half's report. The denominator $J = s_e + s_p - 1$ (Youden's J) also sets how much noise the correction amplifies: a judge barely better than a coin has a small $J$.

In [ ]:
with open("results/summary-v1.json") as handle:
    summary = json.load(handle)
for name, row in summary["correctionOutOfSample"].items():
    J = row["calibratedCatchWrong"] + row["calibratedPassRight"] - 1
    print(
        f"{name:36s} J={J:.3f}  held-out truth {row['heldOutTruth']:.1%}  reported {row['heldOutReported']:.1%}  corrected {row['heldOutCorrected']:.1%}"
    )

In [ ]:
# The paired test: the same 500 answers, judged two ways.
for judge in ["claude-haiku-4-5-20251001", "claude-sonnet-5-5"]:
    right = {
        cond: {
            j["id"]: (j["verdict"] == "INCORRECT") == cands[j["id"]]["truthWrong"]
            for j in judgments
            if j["judge"] == judge and j["condition"] == cond
        }
        for cond in ["direct", "reasoned"]
    }
    b = sum(right["direct"][i] and not right["reasoned"][i] for i in ids)
    c_ = sum(right["reasoned"][i] and not right["direct"][i] for i in ids)
    print(
        f"{judge[:16]}: only one-word right {b}, only reasoned right {c_}, exact McNemar p = {S.mcnemar_exact(b, c_):.3g}"
    )

In [ ]:
# The intervals side by side for the hardest cell: one-word Haiku passing right answers.
import matplotlib.pyplot as plt

got = {j["id"]: j["verdict"] for j in judgments if j["judge"].startswith("claude-haiku") and j["condition"] == "direct"}
c = S.confusion(truth, [got[i] == "INCORRECT" for i in ids])
k, n = c.tn, c.tn + c.fp
methods = {"Wald": S.wald(k, n), "Wilson": S.wilson(k, n), "Clopper-Pearson": S.clopper_pearson(k, n)}
fig, ax = plt.subplots(figsize=(7, 2.2))
for y, (name, (lo, hi)) in enumerate(methods.items()):
    ax.plot([lo, hi], [y, y], lw=6)
    ax.plot([k / n], [y], "ko")
ax.set_yticks(range(3), list(methods))
ax.set_xlim(0, 1)
ax.set_xlabel(f"share of right answers passed ({k}/{n})")
plt.tight_layout()
plt.show()

## Part 2: rerun it on your own key (optional)

Add `ANTHROPIC_API_KEY` in Colab's Secrets panel (the key icon), then run the next cell. It samples 50 problems and stops at a $1 ceiling; the full run in the article used 500 and cost $2.34 at list price. Your numbers will differ from the recorded ones by sampling noise; the intervals say how much.

In [ ]:
try:
    from google.colab import userdata
except ImportError:  # outside Colab: set ANTHROPIC_API_KEY in the environment instead
    userdata = None
if userdata is not None:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.makedirs("mine", exist_ok=True)
!cp judge_calibration_v1.py judge_stats_v1.py mine/ && mkdir -p mine/data mine/results mine/receipts
!cd mine && python judge_calibration_v1.py all --n 50 --max-usd 1